# EG-VAN: EXPLORATORY ONLY nine-label experiment
This trains on existing, real HAM10000, SIIM–ISIC 2020, and Fanconic images. It is **nonclinical**, **not an exact EG-VAN paper reproduction**, and **does not approve a dataset**. Put the complete current `EG-VAN` directory, including its existing `data/` and `analysis/` image files, in `MyDrive/EG-VAN` before running. No dataset download is performed here. Do not substitute missing files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/EG-VAN')
assert (PROJECT / 'src/nine_class_exploratory.py').is_file(), 'Upload the current project first'
%cd /content/drive/MyDrive/EG-VAN


In [ ]:
import sys, torch, torchvision, cv2, matplotlib
sys.path.insert(0, str(PROJECT / 'src'))
from nine_class_colab import validate_colab_runtime
print(validate_colab_runtime(torch, torchvision))
print('OpenCV:', cv2.__version__, 'Matplotlib:', matplotlib.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime'


## Freeze and check real images
The following steps require the actual 10,234 existing originals. They verify source-image SHA-256, source-defined labels, known groups and real image loading. The exploratory manifest retains `NOT_APPROVED` on every record.

In [ ]:
import subprocess
subprocess.run([sys.executable, 'src/nine_class_exploratory.py', 'prepare'], check=True)
subprocess.run([sys.executable, 'src/nine_class_exploratory.py', 'preflight'], check=True)


## Train (25 planned epochs)
Uses the existing T4 config: 384 px, microbatch 8, accumulation 4, FP16 autocast/GradScaler, frozen BatchNorm running statistics, focal loss and Adamax. Each run writes `last.pt`, validation-selected `best.pt`, and `history.csv` directly to Drive. Training may take hours.

In [ ]:
subprocess.run([sys.executable, 'src/nine_class_exploratory.py', 'train'], check=True)


## Resume after interruption
Run this cell only after locating the actual run folder printed during training. Set `RUN_DIR` to that folder; the runner checks the frozen split and config hashes.

In [ ]:
# RUN_DIR = PROJECT / 'analysis/nine_class_exploratory/runs/EXPLORATORY_ONLY_actual_run_id'
# subprocess.run([sys.executable, 'src/nine_class_exploratory.py', 'train', '--resume', str(RUN_DIR / 'last.pt')], check=True)


## Evaluate once training is complete
Select the actual run folder. The command evaluates validation, independent calibration and held-out exploratory test partitions using `best.pt`; temperature is fitted on calibration only. It writes predictions, class metrics, confusion matrix, curves, raw-image quality versus error, uncertainty and a concise JSON report. No external validation is claimed.

In [ ]:
runs = sorted((PROJECT / 'analysis/nine_class_exploratory/runs').glob('EXPLORATORY_ONLY_*'))
assert runs, 'No exploratory training run found'
RUN_DIR = runs[-1]
assert (RUN_DIR / 'best.pt').is_file() and (RUN_DIR / 'last.pt').is_file()
assert not (RUN_DIR / 'evaluation').exists(), 'Evaluation exists; select a fresh output folder'
print('Evaluating', RUN_DIR)
subprocess.run([sys.executable, 'src/nine_class_exploratory.py', 'evaluate', '--checkpoint', str(RUN_DIR / 'best.pt'), '--output-dir', str(RUN_DIR / 'evaluation')], check=True)
